# Experiment 15 — Model revision

Addresses the model problems found in the stress test of 21 Oct 2026:

- **15A** a third parameter set that is bistable *without* adaptation at the test baseline and on which the defined therapy works; sweep of adaptation strength α; and the therapy check on the second set (the α = 0.5 ratio of Figure 4A used a reset state that therapy cannot reach).
- **15B** robustness sample around the third set, with one outcome category per set.
- **15C** experimental-gingivitis challenge calibrated so that naive sites develop moderate inflammation; r = 1 and r = 0.03; largest reversible naive response for each host.
- **15D** rechallenge with naive sites matched to treated sites on resting inflammation (bleeding).
- **15E** safe-challenge map (largest non-converting challenge and its excess response) and stopping rules.
- **15F** single versus three-monthly repeated instrumentation (Badersten design), and the visit-to-visit trend of inflammation-free time under debridement at every recall at r = 0.03.
- **15G** window and rechallenge with Weibull response functions.

Every model call goes through the `periomemory` package (the archived implementation), so the numbers come from the same code as the regression tests. The notebook first reproduces values exported by Experiment_14_Revision.

Set `FAST = True` for a smoke test (about 10 minutes); `FAST = False` for the full run. On Colab the export folder is written to Google Drive (`MyDrive/Biosemiotic_Perio`) as the run proceeds, with a checkpoint after every part and every 20 sets of the 15B sample. **If the runtime disconnects, reconnect and choose Run all again**: finished parts are skipped and the run resumes. The folder is zipped at the end.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install the archived model package (skip if already installed)
import importlib, subprocess, sys
try:
    import periomemory  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/CarloGalliUnipr/periodontal-host-memory-model.git"], check=True)

In [ ]:
import json, os, platform, shutil, time, itertools
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import scipy
import periomemory as pm
from periomemory.model import hill
from scipy.optimize import brentq

# ----------------------------------------------------------------- run settings
FAST = False                # True: smoke test with reduced grids (about 10 min)
TARGET_NAIVE_PEAK = 0.30    # naive peak H that stands for experimental gingivitis (15C)
RUN_15G = True              # Weibull response functions
SEED = 20261021

# ----------------------------------------------------------------- where results are saved
# On Colab the export folder lives in Google Drive and is written as the run proceeds: every finished
# part leaves a marker (_DONE_*) and the 15B sample saves every 20 sets. If the runtime disconnects,
# reconnect and "Run all" again: finished parts are loaded from Drive and skipped, and 15B resumes.
USE_DRIVE = True            # False: write to the Colab disk only (lost on disconnection)
DRIVE_DIR = "/content/drive/MyDrive/Biosemiotic_Perio"
RESUME = True               # False: delete any previous export and start from scratch
NAME = "Experiment_15_Model_Revision_EXPORT" + ("_FAST" if FAST else "")
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB and USE_DRIVE:
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
    OUT = os.path.join(DRIVE_DIR, NAME)
else:
    OUT = NAME
if os.path.exists(OUT) and not RESUME:
    shutil.rmtree(OUT)
os.makedirs(OUT, exist_ok=True)
print("export folder:", os.path.abspath(OUT))
CHECKS = []
T0 = time.time()
_START = [0]

def done(part, tag):
    return os.path.exists(os.path.join(OUT, part, f"_DONE_{tag}"))

def start_part():
    _START[0] = len(CHECKS)

def mark_done(part, tag):
    """Save the checks of this part and leave a completion marker (written last)."""
    new = CHECKS[_START[0]:]
    pd.DataFrame(new, columns=["module", "check", "value", "passed"]).to_csv(
        os.path.join(part_dir(part), f"checks_{tag}.csv"), index=False)
    for c in new:
        c["saved"] = True
    with open(os.path.join(part_dir(part), f"_DONE_{tag}"), "w") as f:
        f.write(time.strftime("%Y-%m-%d %H:%M:%S"))
    print(f"[checkpoint] {part} {tag} saved")

def load(part, fname):
    return pd.read_csv(os.path.join(OUT, part, fname))

def part_dir(name):
    d = os.path.join(OUT, name); os.makedirs(d, exist_ok=True); return d

def save(df, part, fname):
    df.to_csv(os.path.join(part_dir(part), fname), index=False)
    return df

def check(part, what, value, passed=None):
    CHECKS.append(dict(module=part, check=what, value=value, passed=passed, saved=False))
    print(f"[{part}] {what}: {value}" + ("" if passed is None else f"  -> {'OK' if passed else 'FAILED'}"))

# ----------------------------------------------------------------- parameter sets
REF = dict(pm.REFERENCE)                                  # n=q=4, K=0.9, th0=0.8, r=1, a=1
SECOND = dict(pm.SECOND)                                  # n=q=4, K=0.5, th0=0.6
THIRD = dict(n=4.0, q=4.0, K=0.7, th0=0.55, r=1.0, a=0.5) # bistable at the baseline without adaptation
UB_REF, UB_THIRD = 0.20, 0.25
DAYS = np.array([0, 15, 30, 60, 90, 180, 365], float) if not FAST else np.array([0, 30, 90, 365], float)
EPS_LIST = [0.03, 0.01]
NBIS = 16 if not FAST else 11

PARAMS = dict(FAST=FAST, TARGET_NAIVE_PEAK=TARGET_NAIVE_PEAK, SEED=SEED, RUN_15G=RUN_15G)
pfile = os.path.join(OUT, "run_settings.json")
if os.path.exists(pfile):
    prev = json.load(open(pfile))
    if prev != PARAMS:
        raise RuntimeError(f"The export folder holds a run with different settings {prev}; "
                           "set RESUME = False to start again, or restore the settings.")
json.dump(PARAMS, open(pfile, "w"))
with open(os.path.join(OUT, "parameters.json"), "w") as f:
    json.dump(dict(FAST=FAST, TARGET_NAIVE_PEAK=TARGET_NAIVE_PEAK, SEED=SEED, REF=REF, SECOND=SECOND,
                   THIRD=THIRD, UB_REF=UB_REF, UB_THIRD=UB_THIRD, DAYS=DAYS.tolist(), EPS_LIST=EPS_LIST,
                   K_ACCESS=pm.K_ACCESS, T_PC=pm.T_PC, UNIT_DAYS=pm.UNIT_DAYS), f, indent=1)
with open(os.path.join(OUT, "software_versions.txt"), "w") as f:
    f.write(f"python {platform.python_version()}\nnumpy {np.__version__}\nscipy {scipy.__version__}\n"
            f"pandas {pd.__version__}\nmatplotlib {matplotlib.__version__}\nperiomemory {pm.__version__}\n")

## Shared protocol functions

`eg_challenge` generalizes the 21-day experimental-gingivitis challenge of 14A: plaque rises from the baseline towards a plateau U at the regrowth rate of accessible plaque (0.5 per unit) for 21 days, then returns to the baseline; peak H is taken over the challenge window and persistence is H > 0.5 after settling. It is vectorized over sites (each site can have its own onset, plateau and baseline) and optionally applies a stopping rule: as soon as H exceeds `stop`, plaque is removed (u = 0 for one week, then baseline), with or without debridement (M → 0).

In [ ]:
DT = pm.DT
DUR21 = pm.align(pm.days_to_units(21.0))

def model(p, eps, instant=False, **kw):
    return pm.Model(**p, eps=eps, instant=instant, **kw)

def settle_time(eps, instant):
    return 200.0 if instant else max(10.0 / eps, 200.0)

def naive_states(p, ub, N=1, b=0.0):
    s = pm.eq_state(p, ub, "low", gamma=0.0, b=b)
    return pm.tile(s, N)

def treated_states(m, p, ub, N=1, t_pc=pm.T_PC, b=0.0):
    """Diseased equilibrium at ub, then debridement and t_pc units of perfect plaque control.
    Returns (states, H at end of plaque control, fraction of A retained); None if no diseased state."""
    hi = pm.eq_state(p, ub, "high", gamma=0.0, b=b)
    if hi is None:
        return None
    st, _, ret = pm.therapy(m, pm.tile(hi, N), t_pc)
    return st, st[0].copy(), ret

def eg_challenge(m, states, ub, t_on, U, dur=DUR21, ka=pm.K_ACCESS, settle=0.0,
                 stop=None, stop_debride=False, chunk=1.0):
    """Vectorized challenge. ub, t_on, U: scalars or arrays of the number of sites.
    Returns dict with peak (window), peak_any (window + 2 weeks), persistent, stopped, final state."""
    H, M, A, D = [np.array(x, float) for x in states]
    N = H.size
    ub = np.broadcast_to(np.asarray(ub, float), (N,)).copy()
    t_on = pm.align(np.broadcast_to(np.asarray(t_on, float), (N,)).copy())
    U = np.broadcast_to(np.asarray(U, float), (N,)).copy()
    sched = pm.schedule(ub, t_on, dur, U, ka=ka)
    stopped = np.zeros(N, bool); t_stop = np.full(N, np.inf)
    def u_fun(t):
        base = sched(t)
        pc = stopped & (t < t_stop + pm.T_PC - 1e-9)
        return np.where(pc, 0.0, np.where(stopped, ub, base))
    peak = np.zeros(N); peak_any = np.zeros(N)
    def mon(t, H_, M_, A_, D_):
        win = (t > t_on + 1e-9) & (t <= t_on + dur + 1e-9)
        np.maximum(peak, np.where(win, H_, 0.0), out=peak)
        win2 = (t > t_on + 1e-9) & (t <= t_on + dur + pm.days_to_units(14) + 1e-9)
        np.maximum(peak_any, np.where(win2, H_, 0.0), out=peak_any)
    t, t_end = 0.0, float((t_on + dur).max() + pm.days_to_units(14))
    st = (H, M, A, D)
    while t < t_end - 1e-9:
        t1 = min(t + chunk, t_end)
        st, _ = m.run(st, u_fun, t, t1, monitor=mon)
        t = t1
        if stop is not None:
            win = (t > t_on) & (t <= t_on + dur + 1e-9)
            new = win & ~stopped & (st[0] > stop)
            if new.any():
                stopped |= new; t_stop = np.where(new, t, t_stop)
                if stop_debride:
                    st = (st[0], np.where(new, 0.0, st[1]), st[2], st[3])
    # finish any plaque-control period, then settle at baseline
    if settle > 0:
        st, _ = m.run(st, u_fun, t, t + pm.T_PC + 0.0)
        st, _ = m.run(st, lambda tt: ub, 0.0, settle)
    return dict(peak=peak, peak_any=peak_any, persistent=st[0] > 0.5, stopped=stopped, state=st)

def advance(m, state, ub, times):
    """Snapshots of a single-site state run at constant baseline ub, at the given times (units)."""
    order = np.argsort(times); snaps = [None] * len(times); t = 0.0; st = state
    for i in order:
        if times[i] > t:
            st, _ = m.run(st, pm.const(ub), t, pm.align(times[i])); t = pm.align(times[i])
        snaps[i] = tuple(np.array(x, float).copy() for x in st)
    return tuple(np.concatenate([s[k] for s in snaps]) for k in range(4))

def relapse_ceiling(p, eps, instant, ubs, b=0.0):
    """Lowest baseline from which every higher baseline relapses without challenge after the defined therapy."""
    m = model(p, eps, instant)
    rel = []
    for ub in ubs:
        tr = treated_states(m, p, ub, b=b)
        if tr is None:
            rel.append(np.nan); continue
        st, _ = m.run(tr[0], pm.const(ub), 0.0, settle_time(eps, instant))
        rel.append(float(st[0][0] > 0.5))
    rel = np.array(rel)
    for i in range(len(ubs)):
        if np.all(rel[i:] == 1.0):
            return ubs[i], rel
    return np.nan, rel

def crit_grid(m, states, ub, t_on, dur, settle, grid):
    """Critical step challenge on a grid (for hosts where monotonicity is not guaranteed).
    states: single-site; t_on: array of onsets. Returns crit per onset and a monotonicity flag."""
    nT, nG = len(t_on), len(grid)
    st = tuple(np.repeat(x, nT * nG) for x in states)
    tt = np.repeat(pm.align(t_on), nG); UU = np.tile(grid, nT)
    t_end = float((tt + dur).max() + settle)
    out, _ = m.run(st, pm.schedule(ub, tt, pm.align(dur), UU), 0.0, t_end)
    conv = (out[0] > 0.5).reshape(nT, nG)
    crit, mono = [], True
    for row in conv:
        if row.any():
            i = np.argmax(row); crit.append(grid[i]); mono &= bool(row[i:].all())
        else:
            crit.append(np.nan)
    return np.array(crit), mono

def window(p, eps, ub, rec_eT, b=0.0, monotone=True, grid=None):
    """Critical challenge (duration 2/eps, settle 10/eps) after recovery eps*T for naive and treated sites."""
    m = model(p, eps)
    tr = treated_states(m, p, ub, b=b)
    nv = naive_states(p, ub, b=b)
    t_on = np.asarray(rec_eT, float) / eps
    res = {}
    for name, st0 in (("naive", nv), ("treated", None if tr is None else tr[0])):
        if st0 is None:
            res[name] = np.full(len(t_on), np.nan); continue
        if monotone:
            st = tuple(np.repeat(x, len(t_on)) for x in st0)
            c, _ = pm.critical_challenge(m, st, ub, t_on, np.full(len(t_on), 2 / eps), 10 / eps, n_iter=NBIS)
        else:
            c, mono = crit_grid(m, st0, ub, t_on, 2 / eps, 10 / eps, grid)
            check("window", f"conversion monotone on grid ({name}, a={p['a']}, eps={eps})", mono, mono)
        res[name] = c
    return res

## 0. Anchors: reproduce Experiment_14_Revision

In [ ]:
m = model(REF, 0.01)
tr, H_end, ret = treated_states(m, REF, UB_REF)
check("anchor", "A retained after therapy, eps=0.01 (14A: 0.970567)", round(float(ret[0]), 6),
      abs(ret[0] - 0.970567) < 1e-5)
t_on = pm.align(pm.days_to_units(np.array([90.0])))
r_ = eg_challenge(m, tr, UB_REF, t_on, 0.40)
check("anchor", "treated peak H at 90 days, U=0.40, eps=0.01 (14A: 0.527179)", round(float(r_["peak"][0]), 6),
      abs(r_["peak"][0] - 0.527179) < 1e-5)
r_ = eg_challenge(m, naive_states(REF, UB_REF), UB_REF, t_on, 0.40)
check("anchor", "naive peak H, U=0.40, eps=0.01 (14A: 0.060207)", round(float(r_["peak"][0]), 6),
      abs(r_["peak"][0] - 0.060207) < 1e-5)
w = window(REF, 0.03, UB_REF, [0.0])
check("anchor", "critical challenge at eps=0.03, 2/eps: naive (14B: 0.564890)", round(float(w["naive"][0]), 4),
      abs(w["naive"][0] - 0.564890) < 2e-3)
check("anchor", "critical challenge at eps=0.03, 2/eps: treated (14B: 0.294977)", round(float(w["treated"][0]), 4),
      abs(w["treated"][0] - 0.294977) < 2e-3)

## 15A. Bistability that does not depend on adaptation

**A1.** Therapy on the second set: for α ≥ 0.5 the diseased state persists at u = 0 and debridement with plaque control cannot restore the site, so the treated states used for Figure 4A at α = 0.5 were reachable only by the instantaneous reset.

**A2.** Selection scan for a third set: bistable at α = 0 with a positive lower fold, and with the defined therapy succeeding at the baseline. Records, for each candidate, the largest α at which therapy works.

**A3.** On the third set (baseline 0.25), sweep α from −0.5 to 0.5: folds, therapy success, the post-therapy window (critical challenge, 2/ε) against recovery, the relapse ceiling, the ratchet margin and the 21-day rechallenge. α = 0 is the internal control (treated = naive).

In [ ]:
P = "15A_intrinsic_bistability"
if not done(P, "A1"):
    start_part()
    rows = []
    for a in [0.0, 0.25, 0.5, 0.75, 1.0]:
        p = dict(SECOND, a=a)
        for eps, inst in [(0.03, False), (0.01, False), (0.03, True)]:
            mm = model(p, eps, inst)
            for tpc in [4, 10, 40]:
                tr = treated_states(mm, p, 0.25, t_pc=tpc)
                st, _ = mm.run(tr[0], pm.const(0.25), 0.0, settle_time(eps, inst))
                rows.append(dict(alpha=a, host="instant" if inst else f"eps={eps}", plaque_control_units=tpc,
                                 H_after_plaque_control=float(tr[1][0]), restored=bool(st[0][0] < 0.5)))
    A1 = save(pd.DataFrame(rows), P, "A1_second_set_therapy.csv")
    print(A1.pivot_table(index=["alpha", "host"], columns="plaque_control_units", values="restored"))
    check(P, "second set: defined therapy restores the site at alpha=0.5 in any host",
          bool(A1[(A1.alpha == 0.5)].restored.any()))
    mark_done(P, "A1")

In [ ]:
if not done(P, "A2"):
    start_part()
    # A2: selection scan (cheap fold computations, then therapy at the candidate baseline)
    Hg = np.linspace(1e-6, 1 - 1e-6, 20001)
    def folds_fast(n, q, K, th0, a):
        u = th0 * np.exp(-a * Hg) * (Hg / (1 - Hg)) ** (1 / n) - hill(Hg, K, q); du = np.diff(u)
        i = np.where(np.sign(du[1:]) != np.sign(du[:-1]))[0] + 1
        return None if len(i) < 2 else (u[i[0]], u[i[1]])

    ngrid = [3, 4, 5]; Kg = np.round(np.arange(0.40, 0.951, 0.05 if not FAST else 0.1), 3)
    thg = np.round(np.arange(0.40, 0.951, 0.05 if not FAST else 0.1), 3)
    rows = []
    for n, q, K, th0 in itertools.product(ngrid, ngrid, Kg, thg):
        f0 = folds_fast(n, q, K, th0, 0.0)
        if f0 is None:
            continue
        up0, dn0 = f0
        if not (0.03 < dn0 < 0.2 and up0 - dn0 > 0.15 and up0 < 0.7):
            continue
        best = 0.0
        for a in [0.25, 0.5, 0.75, 1.0]:
            fa = folds_fast(n, q, K, th0, a)
            if fa is None:
                break
            ub = 0.5 * (dn0 + min(up0, fa[0]))
            p = dict(n=float(n), q=float(q), K=float(K), th0=float(th0), r=1.0, a=a)
            ok = True
            for eps in EPS_LIST:
                mm = model(p, eps); tr = treated_states(mm, p, ub)
                if tr is None:
                    ok = False; break
                st, _ = mm.run(tr[0], pm.const(ub), 0.0, 300.0)
                ok &= bool(st[0][0] < 0.5)
            if not ok:
                break
            best = a
        rows.append(dict(n=n, q=q, K=K, th0=th0, u_down_a0=dn0, u_up_a0=up0, largest_alpha_therapy_works=best))
    A2 = save(pd.DataFrame(rows), P, "A2_selection_scan.csv")
    print(A2.largest_alpha_therapy_works.value_counts().sort_index())
    check(P, "candidates intrinsically bistable with positive lower fold", len(A2))
    check(P, "of these, therapy works at alpha=1", int((A2.largest_alpha_therapy_works >= 1.0).sum()))
    mark_done(P, "A2")
f0 = pm.folds(THIRD, a=0.0); fa = pm.folds(THIRD)
check(P, "third set folds at alpha=0 (u_down, u_up)", (round(float(f0[3]), 4), round(float(f0[1]), 4)),
      bool(f0[3] < UB_THIRD < f0[1]))
check(P, "third set folds at alpha=0.5 (u_down, u_up)", (round(float(fa[3]), 4), round(float(fa[1]), 4)),
      bool(UB_THIRD < fa[1]))

In [ ]:
if not done(P, "A3_window"):
    start_part()
    # A3: alpha sweep on the third set
    alphas = [-0.5, -0.25, 0.0, 0.125, 0.25, 0.375, 0.5] if not FAST else [-0.25, 0.0, 0.25, 0.5]
    recs = [0.0, 0.25, 0.5, 1.0, 2.0, 4.0] if not FAST else [0.0, 1.0]
    grid = np.round(np.arange(UB_THIRD + 0.01, 4.0, 0.01 if not FAST else 0.05), 3)
    rows, crow, wrows = [], [], []
    for a in alphas:
        p = dict(THIRD, a=a)
        f = pm.folds(p)
        for eps in EPS_LIST:
            mm = model(p, eps); tr = treated_states(mm, p, UB_THIRD)
            if tr is None:
                rows.append(dict(alpha=a, eps=eps, diseased_state=False)); continue
            st, _ = mm.run(tr[0], pm.const(UB_THIRD), 0.0, settle_time(eps, False))
            restored = bool(st[0][0] < 0.5)
            rows.append(dict(alpha=a, eps=eps, diseased_state=True, u_up=None if f is None else f[1],
                             u_down=None if f is None else f[3], A_retained=float(tr[2][0]),
                             restored_after_therapy=restored))
            if not restored:
                continue
            w = window(p, eps, UB_THIRD, recs, monotone=(a >= 0), grid=grid)
            for i, e in enumerate(recs):
                wrows.append(dict(alpha=a, eps=eps, rec_memory_units=e, crit_naive=w["naive"][i],
                                  crit_treated=w["treated"][i], ratio=w["treated"][i] / w["naive"][i]))
        # instant host (shares equilibria): critical challenge after therapy, duration 20
        mi = model(p, 0.03, True)
        tri = treated_states(mi, p, UB_THIRD)
        if tri is not None:
            st = tuple(np.concatenate([a_, b_]) for a_, b_ in zip(naive_states(p, UB_THIRD), tri[0]))
            if a >= 0:
                c, _ = pm.critical_challenge(mi, st, UB_THIRD, np.zeros(2), np.full(2, 20.0), 200.0, n_iter=NBIS)
            else:
                c = [crit_grid(mi, tuple(x[[k]] for x in st), UB_THIRD, np.zeros(1), 20.0, 200.0, grid)[0][0] for k in (0, 1)]
            crow.append(dict(alpha=a, crit_instant_naive=c[0], crit_instant_treated=c[1]))
    A3a = save(pd.DataFrame(rows), P, "A3_therapy_and_folds.csv")
    A3w = save(pd.DataFrame(wrows), P, "A3_window_vs_recovery.csv")
    A3i = save(pd.DataFrame(crow), P, "A3_instant_host.csv")
    print(A3w[A3w.rec_memory_units == 0].to_string())
    if len(A3w):
        a0 = A3w[(A3w.alpha == 0.0) & (A3w.rec_memory_units == 0)]
        check(P, "alpha=0 control: treated/naive ratio equals 1", a0.ratio.round(3).tolist(),
              bool(np.allclose(a0.ratio, 1.0, atol=0.01)))
        pos = A3w[(A3w.alpha > 0) & (A3w.rec_memory_units == 0)]
        check(P, "sensitizing hosts: treated/naive < 1 immediately after therapy", pos.ratio.round(3).tolist(),
              bool((pos.ratio < 1).all()))
    mark_done(P, "A3_window")
else:
    A3w = load(P, "A3_window_vs_recovery.csv")

In [ ]:
if not done(P, "A3_rest"):
    start_part()
    # A3 continued: relapse ceiling, ratchet margin and rechallenge on the third set (alpha = 0.5)
    ubs = np.round(np.arange(0.15, (fa[1] if fa else 0.35) + 1e-9, 0.005 if not FAST else 0.02), 3)
    rows = []
    for eps in [0.1, 0.03, 0.01]:
        c, _ = relapse_ceiling(THIRD, eps, False, ubs)
        rows.append(dict(host=f"eps={eps}", relapse_ceiling=c, static_fold=fa[1], ceiling_over_fold=c / fa[1]))
    c, _ = relapse_ceiling(THIRD, 0.03, True, ubs)
    rows.append(dict(host="instant", relapse_ceiling=c, static_fold=fa[1], ceiling_over_fold=c / fa[1]))
    save(pd.DataFrame(rows), P, "A3_relapse_ceiling.csv"); print(pd.DataFrame(rows))

    peaks = np.round(np.arange(0.20, 0.90, 0.01 if not FAST else 0.02), 3)
    thr_i = pm.ratchet_threshold(THIRD, 0.1, True, 60.0, peaks, 20)
    thr_m = pm.ratchet_threshold(THIRD, 0.01, False, 60.0, peaks, int(max(40, np.ceil(40 / 0.6))) if not FAST else 40)
    save(pd.DataFrame([dict(T=60, instant=thr_i, eps001=thr_m, margin=thr_m / thr_i, static_fold=fa[1])]),
         P, "A3_ratchet_margin.csv")
    check(P, "third set ratchet thresholds at T=60 (instant, eps=0.01)", (float(thr_i), float(thr_m)))

    rows = []
    U_grid = np.round(np.arange(0.30, 0.80, 0.05), 3)
    for eps, inst in [(0.03, False), (0.01, False), (0.03, True)]:
        mm = model(THIRD, eps, inst)
        tr = treated_states(mm, THIRD, UB_THIRD)
        nT, nU = len(DAYS), len(U_grid)
        for hist, st0 in (("naive", naive_states(THIRD, UB_THIRD)), ("treated", tr[0])):
            st = tuple(np.repeat(x, nT * nU) for x in st0)
            tt = np.repeat(pm.days_to_units(DAYS), nU); UU = np.tile(U_grid, nT)
            r_ = eg_challenge(mm, st, UB_THIRD, tt, UU, settle=settle_time(eps, inst))
            for k in range(nT * nU):
                rows.append(dict(host="instant" if inst else f"eps={eps}", history=hist, days=tt[k] * pm.UNIT_DAYS,
                                 U=UU[k], peak_H=r_["peak"][k], persistent=bool(r_["persistent"][k])))
    A3r = save(pd.DataFrame(rows), P, "A3_rechallenge.csv")
    print(A3r[A3r.U == 0.40].pivot_table(index=["host", "history"], columns="days", values="peak_H").round(2))
    mark_done(P, "A3_rest")

## 15B. Robustness sample around the third set

Latin-hypercube sample (n, q 3–5; K 0.55–0.85; θ0 0.45–0.65; α 0.25–0.5). For each set: folds at α = 0 and at α; baseline at the midpoint between the α = 0 lower fold and the smaller of the two upper folds, so that the baseline lies inside the bistable range *without* adaptation. One outcome category per set: not intrinsically bistable, no diseased state at baseline, therapy fails (relapse without challenge after the defined therapy), naive never converts, evaluable. Window ratios at ε = 0.03, duration 2/ε.

In [ ]:
P = "15B_intrinsic_sample"
NS = 300 if not FAST else 12
rng = np.random.default_rng(SEED)
lo = np.array([3, 3, 0.55, 0.45, 0.25]); hi_ = np.array([5, 5, 0.85, 0.65, 0.50])
u = (rng.permuted(np.tile(np.arange(NS), (5, 1)), axis=1).T + rng.random((NS, 5))) / NS
X = lo + u * (hi_ - lo)
partial = os.path.join(part_dir(P), "B_sample_results_partial.csv")
rows = pd.read_csv(partial).to_dict("records") if os.path.exists(partial) else []
done_ids = {int(r_["set_id"]) for r_ in rows}
if done_ids:
    print(f"resuming 15B: {len(done_ids)} of {NS} sets already done")
for k, (n, q, K, th0, a) in enumerate(X):
    if k in done_ids:
        continue
    if rows and len(rows) % 20 == 0:
        pd.DataFrame(rows).to_csv(partial, index=False)
    p = dict(n=n, q=q, K=K, th0=th0, r=1.0, a=a)
    row = dict(set_id=k, n=n, q=q, K=K, th0=th0, a=a)
    f0, fa_ = pm.folds(p, a=0.0), pm.folds(p)
    if f0 is None or f0[3] <= 0 or fa_ is None:
        row["category"] = "not intrinsically bistable at u > 0"; rows.append(row); continue
    ub = 0.5 * (f0[3] + min(f0[1], fa_[1]))
    row.update(u_down_a0=f0[3], u_up_a0=f0[1], u_down=fa_[3], u_up=fa_[1], baseline=ub)
    m3 = model(p, 0.03); tr = treated_states(m3, p, ub)
    if tr is None:
        row["category"] = "no diseased state at baseline"; rows.append(row); continue
    st, _ = m3.run(tr[0], pm.const(ub), 0.0, settle_time(0.03, False))
    row["A_retained"] = float(tr[2][0])
    if st[0][0] > 0.5:
        row["category"] = "therapy fails (relapse without challenge)"; rows.append(row); continue
    w = window(p, 0.03, ub, [0.0])
    mi = model(p, 0.03, True)
    ci, _ = pm.critical_challenge(mi, naive_states(p, ub), ub, np.zeros(1), np.full(1, 20.0), 200.0, n_iter=NBIS)
    row.update(crit_naive=w["naive"][0], crit_treated=w["treated"][0], crit_instant=ci[0])
    if np.isnan(w["naive"][0]):
        row["category"] = "naive never converts"; rows.append(row); continue
    row.update(ratio_treated_naive=w["treated"][0] / w["naive"][0], ratio_treated_instant=w["treated"][0] / ci[0])
    row["category"] = "evaluable"; rows.append(row)
pd.DataFrame(rows).to_csv(partial, index=False)
B = save(pd.DataFrame(rows).sort_values("set_id"), P, "B_sample_results.csv")
print(B.category.value_counts())
ev = B[B.category == "evaluable"]
if len(ev):
    q10, q50, q90 = np.percentile(ev.ratio_treated_naive, [10, 50, 90])
    check(P, f"treated/naive median [10th, 90th] (n={len(ev)})", (round(float(q50), 3), round(float(q10), 3), round(float(q90), 3)))
    check(P, "all evaluable sets treated/naive < 1", bool((ev.ratio_treated_naive < 1).all()),
          bool((ev.ratio_treated_naive < 1).all()))
save(B.category.value_counts().rename_axis("category").reset_index(name="sets"), P, "B_categories.csv")

## 15C. Calibrated experimental gingivitis

For each host the plateau U\* is found at which a naive site reaches peak H = TARGET_NAIVE_PEAK during the 21-day challenge. Treated sites are then challenged with U\* at increasing times after therapy. The instant host receives the memory hosts' U\* and its own. Also recorded: the largest naive peak a challenge can produce without leaving a naive site persistently inflamed (experimental gingivitis is reversible in healthy people). Reference set (baseline 0.20) and third set (baseline 0.25), at r = 1 and r = 0.03.

In [ ]:
P = "15C_calibrated_gingivitis"
if not done(P, "C"):
    start_part()

    def calibrate(m, p, ub, target, lo_=None, hi_=2.0, it=30, nv=None):
        nv = naive_states(p, ub) if nv is None else nv; lo_ = ub if lo_ is None else lo_
        pk = lambda U: eg_challenge(m, nv, ub, 0.0, U)["peak"][0]
        if pk(hi_) < target:
            return np.nan
        for _ in range(it):
            mid = 0.5 * (lo_ + hi_)
            lo_, hi_ = (mid, hi_) if pk(mid) < target else (lo_, mid)
        return hi_

    rows, rev = [], []
    Ugrid = np.round(np.arange(0.22, 1.2, 0.01 if not FAST else 0.04), 3)
    for setname, p0, ub in [("reference", REF, UB_REF), ("third", THIRD, UB_THIRD)]:
        for r in [1.0, 0.03]:
            p = dict(p0, r=r)
            Ustar = {}
            for eps, inst in [(0.03, False), (0.01, False), (0.03, True)]:
                host = "instant" if inst else f"eps={eps}"
                mm = model(p, eps, inst)
                Ustar[host] = calibrate(mm, p, ub, TARGET_NAIVE_PEAK)
                # largest reversible naive response
                Ug_ = Ugrid[Ugrid > ub + 1e-9]
                nv = naive_states(p, ub, len(Ug_))
                rr = eg_challenge(mm, nv, ub, 0.0, Ug_, settle=settle_time(eps, inst))
                okU = ~rr["persistent"]
                rev.append(dict(set=setname, r=r, host=host, U_star=Ustar[host],
                                max_reversible_naive_peak=float(rr["peak"][okU].max()) if okU.any() else np.nan,
                                lowest_U_converting_naive=float(Ug_[rr["persistent"]].min()) if rr["persistent"].any() else np.nan))
            for eps, inst in [(0.03, False), (0.01, False), (0.03, True)]:
                host = "instant" if inst else f"eps={eps}"
                mm = model(p, eps, inst)
                tr = treated_states(mm, p, ub)
                for Ulabel, U in ([("own U*", Ustar[host])] + ([("U* of eps=0.01", Ustar["eps=0.01"])] if inst else [])):
                    if np.isnan(U):
                        continue
                    for hist, st0 in (("naive", naive_states(p, ub)), ("treated", tr[0])):
                        st = tuple(np.repeat(x, len(DAYS)) for x in st0)
                        rr = eg_challenge(mm, st, ub, pm.days_to_units(DAYS), U, settle=settle_time(eps, inst))
                        for k, d in enumerate(DAYS):
                            rows.append(dict(set=setname, r=r, host=host, challenge=Ulabel, U=U, history=hist, days=d,
                                             H_after_therapy=float(tr[1][0]), peak_H=rr["peak"][k],
                                             persistent=bool(rr["persistent"][k])))
    C = save(pd.DataFrame(rows), P, "C_calibrated_rechallenge.csv")
    Crev = save(pd.DataFrame(rev), P, "C_reversibility_and_Ustar.csv")
    print(Crev.to_string())
    print(C[(C.challenge == "own U*")].pivot_table(index=["set", "r", "host", "history"], columns="days",
                                                    values="peak_H").round(2))
    mark_done(P, "C")
else:
    C = load(P, "C_calibrated_rechallenge.csv")
    Crev = load(P, "C_reversibility_and_Ustar.csv")

## 15D. Matching on resting inflammation (bleeding)

Treated sites carry raised resting inflammation after therapy. Here naive sites are brought to the same resting H by raising their baseline plaque (low-branch equilibrium with that H), and both receive (i) the same plateau U or (ii) the same increment U − baseline. If the excess response of treated sites survives, the discriminating test holds when pre-challenge bleeding is matched. Reference set, baseline 0.20, U = 0.40 and the calibrated U\* of 15C.

In [ ]:
P = "15D_matched_bleeding"
if not done(P, "D"):
    start_part()
    rows = []
    Hup = pm.folds(REF)[0]
    for eps in EPS_LIST:
        mm = model(REF, eps)
        tr = treated_states(mm, REF, UB_REF)
        snaps = advance(mm, tr[0], UB_REF, pm.days_to_units(DAYS))
        Hrest = snaps[0]
        Ustar = float(Crev[(Crev.set == "reference") & (Crev.r == 1.0) & (Crev.host == f"eps={eps}")].U_star.iloc[0])
        u_match = np.array([float(pm.u_curve(REF, np.array([h]))[0]) if h < Hup else np.nan for h in Hrest])
        for U in [0.40, Ustar]:
            r_t = eg_challenge(mm, snaps, UB_REF, 0.0, U, settle=settle_time(eps, False))
            nv_std = naive_states(REF, UB_REF, len(DAYS))
            r_n = eg_challenge(mm, nv_std, UB_REF, 0.0, U)
            ok = ~np.isnan(u_match)
            nv_m = tuple(np.concatenate([np.array(pm.eq_state(REF, um, "low", gamma=0.0)[k:k+1]) for um in u_match[ok]])
                         for k in range(4)) if ok.any() else None
            stl = settle_time(eps, False)
            r_same = eg_challenge(mm, nv_m, u_match[ok], 0.0, U, settle=stl) if ok.any() else None
            r_incr = eg_challenge(mm, nv_m, u_match[ok], 0.0, u_match[ok] + (U - UB_REF), settle=stl) if ok.any() else None
            j = 0
            for k, d in enumerate(DAYS):
                row = dict(eps=eps, U=U, days=d, H_rest_treated=Hrest[k], H_rest_naive=float(nv_std[0][0]),
                           bleeding_per_plaque_treated=Hrest[k] / UB_REF, bleeding_per_plaque_naive=float(nv_std[0][0]) / UB_REF,
                           u_matched_naive=u_match[k], peak_treated=r_t["peak"][k], persistent_treated=bool(r_t["persistent"][k]),
                           peak_naive_standard=r_n["peak"][k])
                if ok[k]:
                    row.update(peak_naive_matched_same_U=r_same["peak"][j], persistent_naive_matched_same_U=bool(r_same["persistent"][j]),
                               peak_naive_matched_same_increment=r_incr["peak"][j],
                               persistent_naive_matched_same_increment=bool(r_incr["persistent"][j])); j += 1
                rows.append(row)
    Dm = save(pd.DataFrame(rows), P, "D_matched_bleeding.csv")
    print(Dm.round(3).to_string())
    mark_done(P, "D")
else:
    Dm = load(P, "D_matched_bleeding.csv")

## 15E. Safe-challenge map and stopping rules

For each host, time since therapy and challenge duration (7, 14, 21 days): the largest plateau that never leaves a treated site persistently inflamed (U_safe), and the excess response (treated − naive peak) at U_safe. Stopping rules: plaque removed as soon as H exceeds h_stop (0.2, 0.3, 0.5), with or without debridement. Reference set, baseline 0.20.

In [ ]:
P = "15E_safe_challenge"
if not done(P, "E"):
    start_part()
    Ug = np.round(np.arange(UB_REF + 0.01, 0.71, 0.01 if not FAST else 0.03), 3)
    durs = [7, 14, 21] if not FAST else [7, 21]
    days_e = DAYS
    rows, srows = [], []
    for eps, inst in [(0.03, False), (0.01, False)]:
        mm = model(REF, eps, inst)
        tr = treated_states(mm, REF, UB_REF)
        for dd in durs:
            dur = pm.align(pm.days_to_units(dd))
            nT, nU = len(days_e), len(Ug)
            tt = np.repeat(pm.days_to_units(days_e), nU); UU = np.tile(Ug, nT)
            r_t = eg_challenge(mm, tuple(np.repeat(x, nT * nU) for x in tr[0]), UB_REF, tt, UU, dur=dur,
                               settle=settle_time(eps, inst))
            r_n = eg_challenge(mm, naive_states(REF, UB_REF, nU), UB_REF, 0.0, Ug, dur=dur, settle=settle_time(eps, inst))
            for i, d in enumerate(days_e):
                sl = slice(i * nU, (i + 1) * nU)
                pers = r_t["persistent"][sl]
                safe_idx = (np.argmax(pers) - 1) if pers.any() else nU - 1
                row = dict(host=f"eps={eps}", duration_days=dd, days=d,
                           U_safe=Ug[safe_idx] if safe_idx >= 0 else np.nan,
                           lowest_converting_U=Ug[np.argmax(pers)] if pers.any() else np.nan,
                           naive_converts_any=bool(r_n["persistent"].any()))
                if safe_idx >= 0:
                    row.update(peak_treated_at_U_safe=r_t["peak"][sl][safe_idx], peak_naive_at_U_safe=r_n["peak"][safe_idx],
                               excess_at_U_safe=r_t["peak"][sl][safe_idx] - r_n["peak"][safe_idx])
                rows.append(row)
        # stopping rules (21 days)
        for h_stop, deb in itertools.product([0.2, 0.3, 0.5], [False, True]):
            days_s = np.array([0.0, 30.0, 90.0])
            nT, nU = len(days_s), len(Ug)
            tt = np.repeat(pm.days_to_units(days_s), nU); UU = np.tile(Ug, nT)
            r_s = eg_challenge(mm, tuple(np.repeat(x, nT * nU) for x in tr[0]), UB_REF, tt, UU,
                               settle=settle_time(eps, inst), stop=h_stop, stop_debride=deb)
            for i, d in enumerate(days_s):
                sl = slice(i * nU, (i + 1) * nU)
                pers = r_s["persistent"][sl]
                srows.append(dict(host=f"eps={eps}", days=d, h_stop=h_stop, debridement_at_stop=deb,
                                  lowest_converting_U=Ug[np.argmax(pers)] if pers.any() else np.nan,
                                  any_conversion=bool(pers.any()), fraction_stopped=float(r_s["stopped"][sl].mean()),
                                  max_peak_any=float(r_s["peak_any"][sl].max())))
    E = save(pd.DataFrame(rows), P, "E_safe_map.csv")
    Es = save(pd.DataFrame(srows), P, "E_stopping_rules.csv")
    print(E.round(3).to_string()); print(Es.to_string())
    mark_done(P, "E")

## 15F. Single versus repeated instrumentation, and the visit-to-visit trend

**F1 (Badersten design).** Diseased sites with residual attachment loss D0 (reference set with attachment loss: γ = 0.8, δ = 0.001, h_D = 0.8) receive the defined therapy, then 8 recall intervals (T = 60, about 27 months) of plaque regrowth to a maintenance peak with supragingival cleaning at each recall; either no further debridement or debridement at every recall. Outcomes: inflammation-free fraction of follow-up, resolved at the end, attachment loss gained.

**F2.** Debridement at every recall from a naive start at plaque peaks 0.60 and 0.75 (as in Figure S15A–B), at r = 1 and r = 0.03: inflammation-free fraction of each interval across 40 visits and its trend.

In [ ]:
P = "15F_repeated_instrumentation"
if not done(P, "F"):
    start_part()
    D0s = np.array([0.0, 0.1, 0.2, 0.3, 0.4, 0.5]); mpk = np.array([0.10, 0.20, 0.30, 0.40, 0.50])
    NC = 8; T = 60.0
    rows = []
    for r in [0.03, 1.0]:
        p = dict(REF, r=r)
        for eps, inst in [(0.03, True), (0.03, False), (0.01, False)]:
            mm = model(p, eps, inst, gamma=pm.GAMMA, delta=0.001)
            DD, PK = np.meshgrid(D0s, mpk, indexing="ij"); DD, PK = DD.ravel(), PK.ravel(); N = DD.size
            hi = [pm.eq_state(p, 0.20, "high", D=d) for d in DD]
            st0 = tuple(np.array([h[k] for h in hi]) for k in range(4))
            for arm in ["single", "repeated"]:
                st, _, _ = pm.therapy(mm, st0)
                free = np.zeros(N); steps = [0]
                def mon(t, H, M, A, D_):
                    free[:] += (H < 0.5); steps[0] += 1
                umax = PK / (1 - np.exp(-pm.K_REGROW * T))
                for c in range(NC):
                    if arm == "repeated" and c > 0:
                        st = (st[0], np.zeros(N), st[2], st[3])
                    st, _ = mm.run(st, lambda t: umax * (1 - np.exp(-pm.K_REGROW * t)), 0.0, T, monitor=mon)
                for k in range(N):
                    rows.append(dict(r=r, host="instant" if inst else f"eps={eps}", arm=arm, D0=DD[k], maintenance_peak=PK[k],
                                     inflammation_free_fraction=free[k] / steps[0], resolved_at_end=bool(st[0][k] < 0.5),
                                     attachment_loss_gained=st[3][k] - DD[k]))
    F1 = save(pd.DataFrame(rows), P, "F1_single_vs_repeated.csv")
    piv = F1.pivot_table(index=["r", "host", "maintenance_peak", "D0"], columns="arm", values="inflammation_free_fraction")
    piv["repeated_minus_single"] = piv["repeated"] - piv["single"]
    save(piv.reset_index(), P, "F1_difference.csv"); print(piv.round(2).to_string())

    rows, trows = [], []
    NV = 40 if not FAST else 12
    for r in [1.0, 0.03]:
        p = dict(REF, r=r)
        for eps, inst in [(0.03, True), (0.01, False), (0.003, False)]:
            mm = model(p, eps, inst)
            peaks2 = np.array([0.60, 0.75]); N = 2
            st = tuple(np.zeros(N) for _ in range(4)); umax = peaks2 / (1 - np.exp(-pm.K_REGROW * T))
            fr = []
            for v in range(NV):
                st = (st[0], np.zeros(N), st[2], st[3])
                free = np.zeros(N); cnt = [0]
                def mon(t, H, M, A, D_):
                    free[:] += (H < 0.5); cnt[0] += 1
                st, _ = mm.run(st, lambda t: umax * (1 - np.exp(-pm.K_REGROW * t)), 0.0, T, monitor=mon)
                fr.append(free / cnt[0])
            fr = np.array(fr)
            for j, pk in enumerate(peaks2):
                for v in range(NV):
                    rows.append(dict(r=r, host="instant" if inst else f"eps={eps}", peak=pk, visit=v + 1, free_fraction=fr[v, j]))
                x = np.arange(2, NV + 1); slope = np.polyfit(x, fr[1:, j], 1)[0] if NV > 2 else np.nan
                trows.append(dict(r=r, host="instant" if inst else f"eps={eps}", peak=pk, first_visit=fr[0, j],
                                  mean_last_10=fr[-10:, j].mean(), slope_per_visit=slope))
    F2 = save(pd.DataFrame(rows), P, "F2_debridement_every_recall.csv")
    F2t = save(pd.DataFrame(trows), P, "F2_trend.csv"); print(F2t.round(4).to_string())
    mark_done(P, "F")

## 15G. Weibull response functions

Host and community Hill functions replaced by Weibull curves matched in midpoint, midpoint slope and maximum (m = p / (2 ln 2), as in Robustness R3). Folds, window ratio at 2/ε and the 21-day rechallenge at the plateau U* calibrated as in 15C (naive peak = TARGET_NAIVE_PEAK), with naive comparison.

In [ ]:
P = "15G_weibull"
if not done(P, "G"):
    start_part()
    LN2 = np.log(2.0)
    def weib(x, K, p):
        x = np.maximum(x, 0.0)
        return 1.0 - np.exp(-LN2 * (x / K) ** (p / (2 * LN2)))

    class WeibullModel(pm.Model):
        def deriv(self, H, M, A, D, u):
            if self.instant:
                A = self.target(H)
            th = self.th0 * np.exp(-A)
            dH = -H + weib(M + u, th, self.n)
            dM = self.r * (-M + weib(H, self.K, self.q) + self.gamma * D * (self.c + (1.0 - self.c) * H))
            dA = np.zeros_like(H) if self.instant else self.eps * (self.target(H) - A)
            dD = self.delta * np.maximum(H - self.hD, 0.0) * (1.0 - D)
            return dH, dM, dA, dD

    HG = np.linspace(1e-7, 1 - 1e-7, 200001)
    def u_curve_w(p, H=HG, a=None):
        a = p["a"] if a is None else a
        mn = p["n"] / (2 * LN2)
        return p["th0"] * np.exp(-a * H) * (-np.log(1 - H) / LN2) ** (1 / mn) - weib(H, p["K"], p["q"])
    def folds_w(p, a=None):
        u = u_curve_w(p, a=a); du = np.diff(u)
        i = np.where(np.sign(du[1:]) != np.sign(du[:-1]))[0] + 1
        return None if len(i) < 2 else (HG[i[0]], u[i[0]], HG[i[1]], u[i[1]])
    def eq_state_w(p, ub, branch):
        v = u_curve_w(p) - ub
        i = np.where(np.sign(v[1:]) != np.sign(v[:-1]))[0]
        if not len(i):
            return None
        f = lambda H: u_curve_w(p, np.array([H]))[0] - ub
        roots = [brentq(f, HG[j], HG[j + 1]) for j in i]
        H = roots[0] if branch == "low" else roots[-1]
        return H, weib(np.array(H), p["K"], p["q"]), p["a"] * H, 0.0

    rows = []
    if RUN_15G:
        for setname, p, ub in [("reference", REF, UB_REF), ("third", THIRD, UB_THIRD)]:
            fw = folds_w(p); fw0 = folds_w(p, a=0.0)
            row = dict(set=setname, u_up=None if fw is None else fw[1], u_down=None if fw is None else fw[3],
                       bistable_at_a0=fw0 is not None and fw0[3] < ub < fw0[1])
            lowst, hist = eq_state_w(p, ub, "low"), eq_state_w(p, ub, "high")
            if fw is None or hist is None or not (fw[3] < ub < fw[1]):
                row["note"] = "baseline not bistable with Weibull functions"; rows.append(row); continue
            for eps in EPS_LIST:
                mm = WeibullModel(**p, eps=eps)
                tr, _, ret = pm.therapy(mm, pm.tile(hist, 1))
                st, _ = mm.run(tr, pm.const(ub), 0.0, settle_time(eps, False))
                restored = bool(st[0][0] < 0.5)
                nv = pm.tile(lowst, 1)
                out = dict(row, eps=eps, A_retained=float(ret[0]), restored_after_therapy=restored)
                if restored:
                    both = tuple(np.concatenate([a_, b_]) for a_, b_ in zip(nv, tr))
                    c, _ = pm.critical_challenge(mm, both, ub, np.zeros(2), np.full(2, 2 / eps), 10 / eps, n_iter=NBIS)
                    out.update(crit_naive=c[0], crit_treated=c[1], ratio=c[1] / c[0])
                    U = calibrate(mm, p, ub, TARGET_NAIVE_PEAK, nv=nv)
                    dd = np.array([0.0, 60.0, 180.0])
                    rt = eg_challenge(mm, tuple(np.repeat(x, 3) for x in tr), ub, pm.days_to_units(dd), U)
                    rn = eg_challenge(mm, nv, ub, 0.0, U)
                    out.update(U=U, peak_naive=rn["peak"][0], peak_treated_0d=rt["peak"][0],
                               peak_treated_60d=rt["peak"][1], peak_treated_180d=rt["peak"][2])
                rows.append(out)
    G = save(pd.DataFrame(rows), P, "G_weibull.csv"); print(G.round(3).to_string())
    mark_done(P, "G")

## Figures (quick diagnostics) and export

In [ ]:
figd = part_dir("figures")
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
if len(A3w):
    for a, g in A3w[A3w.eps == 0.03].groupby("alpha"):
        ax[0].plot(g.rec_memory_units, g.ratio, "o-", label=f"α = {a}")
    ax[0].axhline(1, color="k", lw=0.5); ax[0].set_xlabel("recovery (memory times)"); ax[0].set_ylabel("treated / naive")
    ax[0].set_title("15A third set, ε = 0.03"); ax[0].legend(fontsize=7)
cc = C[(C.set == "reference") & (C.r == 1.0) & (C.challenge == "own U*")]
for (h, hist), g in cc.groupby(["host", "history"]):
    ax[1].plot(g.days, g.peak_H, "o-" if hist == "treated" else "o:", label=f"{h} {hist}")
ax[1].set_xlabel("days since therapy"); ax[1].set_ylabel("peak H"); ax[1].set_title("15C calibrated challenge (reference)")
ax[1].legend(fontsize=6)
for eps, g in Dm[Dm.U == 0.40].groupby("eps"):
    ax[2].plot(g.days, g.peak_treated, "o-", label=f"treated ε={eps}")
    if "peak_naive_matched_same_increment" in g:
        ax[2].plot(g.days, g.peak_naive_matched_same_increment, "s:", label=f"naive matched ε={eps}")
ax[2].set_xlabel("days since therapy"); ax[2].set_title("15D matched resting H, U = 0.40"); ax[2].legend(fontsize=6)
fig.tight_layout(); fig.savefig(os.path.join(figd, "fig15_overview.png"), dpi=150)

import glob
saved = [pd.read_csv(f_) for f_ in sorted(glob.glob(os.path.join(OUT, "*", "checks_*.csv")))]
unsaved = pd.DataFrame([c_ for c_ in CHECKS if not c_.get("saved")], columns=["module", "check", "value", "passed"])
pd.concat(saved + [unsaved], ignore_index=True).to_csv(os.path.join(OUT, "all_checks.csv"), index=False)
with open(os.path.join(OUT, "README.txt"), "w") as f:
    f.write("""Experiment 15 - Model revision (see notebook markdown for definitions)
15A_intrinsic_bistability: A1 therapy on the second set; A2 selection scan; A3 third set (n=q=4, K=0.7, th0=0.55,
  baseline 0.25): therapy and folds by alpha, window vs recovery, instant host, relapse ceiling, ratchet margin, rechallenge.
15B_intrinsic_sample: Latin-hypercube sample around the third set with one category per set; window ratios.
15C_calibrated_gingivitis: U* giving naive peak = TARGET_NAIVE_PEAK; rechallenge at U*; reversibility of naive challenges.
15D_matched_bleeding: rechallenge with naive sites matched on resting H (same plateau, same increment).
15E_safe_challenge: largest non-converting plateau by time since therapy and duration; stopping rules.
15F_repeated_instrumentation: single vs repeated instrumentation (F1); debridement at every recall at r = 1 and 0.03 (F2).
15G_weibull: folds, window ratio and rechallenge with Weibull response functions.
all_checks.csv: anchors reproducing Experiment_14_Revision and internal checks.
""")
print(f"elapsed {time.time() - T0:.0f} s")
zip_path = shutil.make_archive(OUT, "zip", os.path.dirname(os.path.abspath(OUT)), os.path.basename(OUT))
print("wrote", zip_path)
if IN_COLAB:
    try:
        from google.colab import files
        files.download(zip_path)
    except Exception as e:
        print("download did not start; the zip is in Drive:", e)